In [7]:
import json
import transformers.utils.import_utils
if not hasattr(transformers.utils.import_utils, 'is_torch_fx_available'):
    transformers.utils.import_utils.is_torch_fx_available = lambda: False

from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, PointStruct, SparseVector
from FlagEmbedding import BGEM3FlagModel
from sentence_transformers import CrossEncoder
from tqdm.notebook import tqdm

client = QdrantClient(host="127.0.0.1", port=6333, timeout=120)
COLLECTION_NAME = "legal_hybrid_base"

C:\Users\asus\AppData\Local\Temp\ipykernel_21904\2971913346.py:12: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  client = QdrantClient(host="127.0.0.1", port=6333, timeout=120)


In [8]:
print("Пересоздаем коллекцию векторов")
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense": VectorParams(size=1024, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams()
    }
)

Пересоздаем коллекцию векторов


C:\Users\asus\AppData\Local\Temp\ipykernel_21904\1571878991.py:2: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


UnexpectedResponse: Unexpected Response: 503 (Service Unavailable)
Raw response content:
b''

In [ ]:
print("Загружаем BGE-M3")
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=False, device='cpu')

print("Загружаем BGE-Reranker-v2-m3")
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=512, device='cpu')

with open("chunked_documents.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Начинаем векторизацию и загрузку {len(chunks)} чанков")

In [ ]:
points = []
for i, chunk in enumerate(tqdm(chunks, desc="Обработка чанков")):
    text = chunk["text"]
    
    output = model.encode(text, return_dense=True, return_sparse=True, return_colbert_vecs=False)
    
    dense_vec = output['dense_vecs'].tolist()
    lexical_weights = output['lexical_weights']
    
    unique_tokens = {}
    for token_str, weight in lexical_weights.items():
        token_id = model.tokenizer.convert_tokens_to_ids(token_str)
        if token_id is not None:
            if token_id in unique_tokens:
                unique_tokens[token_id] = max(unique_tokens[token_id], float(weight))
            else:
                unique_tokens[token_id] = float(weight)
                
    indices = list(unique_tokens.keys())
    values = list(unique_tokens.values())
                
    point = PointStruct(
        id=i,
        vector={
            "dense": dense_vec,
            "sparse": SparseVector(indices=indices, values=values)
        },
        payload={
            "source": chunk["source"],
            "text": text
        }
    )
    points.append(point)

client.upload_points(
    collection_name=COLLECTION_NAME,
    points=points,
    batch_size=50
)

print("Гибридная база создана")